# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/720-hz/flyrank-ml-internship/blob/main/work/notebooks/Week%208/capstone.ipynb?flush_cache=true)

This notebook is the working synthesis behind the deployed paper (`docs/index.html`, live at
`https://720-hz.github.io/flyrank-ml-internship/`). It reproduces the headline numbers directly
from the public, no-credentials-needed sample data (`data/raw/content_refresh_anonymized.csv`) so
a reader can rerun it end to end, and pulls forward the validated methodology from Weeks 1-7
(`w01_research_question.ipynb` through `w07_action_playbook.ipynb`) rather than re-deriving it
from scratch. Nothing here overrides an earlier week's finding — this is a synthesis pass, not a
redo.

## 1. Question

*The research question and the decision it supports.*

**Question:** with a fixed weekly review capacity, which content pages should a strategist look at
first — and does a learned model actually beat a transparent, hand-built rule at ranking them?

**Decision this supports:** which pages a content strategist reviews this week, out of a backlog
far larger than any realistic capacity can clear. **Who acts:** a content strategist / SEO editor
with a capped number of reviews per cycle (this project uses 20 and 50 as illustrative capacities).
**Cost of a wrong call:** a false positive burns one of a reviewer's limited slots on a page that
didn't need it (bounded, recoverable); a false negative lets a page with real demand keep losing
visibility unnoticed until the next cycle (compounding, not neutral). Because reviewer attention is
the scarce resource, this is a **ranking** problem and Precision@K is the metric that matches the
real decision — not raw accuracy. (Full framing: `w01_research_question.ipynb`, `w02_ml_task_framing.ipynb`.)

In [1]:
import os, sys
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
print("Working dir:", os.getcwd())

import pandas as pd
import numpy as np

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
n = len(declining_with_demand)
print(f"{len(df):,} pages | {df['client_id'].nunique()} clients | base decline rate {df['is_declining_label'].mean():.3f}")
print(f"Declining pages with real demand (impressions_90d >= 100): {n:,} "
      f"({n / (df['trend_direction']=='down').sum():.1%} of all declining pages)")
for capacity in [20, 50]:
    print(f"  at {capacity} pages/week reviewer capacity: {n/capacity:.0f} weeks of backlog just to clear today's declining set")

Working dir: /home/claude/flyrank-ml-internship


30,000 pages | 32 clients | base decline rate 0.542
Declining pages with real demand (impressions_90d >= 100): 13,152 (80.9% of all declining pages)
  at 20 pages/week reviewer capacity: 658 weeks of backlog just to clear today's declining set
  at 50 pages/week reviewer capacity: 263 weeks of backlog just to clear today's declining set


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source used for this paper's reproducible results:** `data/raw/content_refresh_anonymized.csv`
— an anonymized sample shipped in the repo, no gated access required. **Grain:** one row per
content item, per client, as of one trailing 90-day snapshot. **Scope:** 30,000 content items,
32 pseudonymous clients (`client_id` / `content_id` are opaque hashes — grouping and joins only,
never features).

**Deliberately excluded, and why:**
- FlyRank product-decision fields (`health_score`, `priority_score`, `action_type`) — not shipped
  in this release; would be circular if they were (they encode a decision, not an observation).
- Raw client names, domains, URLs, titles, or search queries — never shipped, and excluded from
  every output this project produces.

**A separate, exploratory data track (not the reproducible pipeline below):** `w03_data_contract.ipynb`
queried FlyRank's full production warehouse (a ~9.8M-row daily panel for `month=2026-03`, via gated
Hugging Face access) to test whether a genuine forward-observed label was buildable at all. It is —
a 5-feature honest model scored ROC-AUC 0.775 there — but that path needs requested credentials, so
it is named here as important groundwork and disclosed in Limitations, not used as this paper's
headline numbers.

In [2]:
# Sanity confirmation that no client-identifying columns are present in the public sample.
suspicious = [c for c in df.columns if any(k in c.lower() for k in ["name", "url", "domain", "email", "query", "title"])]
print("Columns matching client-identifying keywords:", suspicious or "none")
print("client_id sample (pseudonymous hash, grouping only):", df["client_id"].iloc[0])

Columns matching client-identifying keywords: none
client_id sample (pseudonymous hash, grouping only): client_f369cb89fc


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label.** `is_declining_label = (trend_direction == "down")`, bucketed from `trend_pct` (a
comparison of the most recent 30 days of impressions against the prior 30). This is an **observed,
backward-looking proxy**, not a future-observed outcome — named honestly in Limitations.

**Baseline — a transparent CTR-vs-position rule** (`w04_baseline_score.ipynb`). Two candidate
signals were tested before picking one: staleness (days since last update) came back **MIXED** —
decline rate does not climb monotonically with page age. CTR vs. position tier came back
**CONFIRMED** — mean CTR falls cleanly from 2.71% at top-3 positions to 0.15% at deep positions.
The rule is built on the confirmed signal only: `score = visible × reachable × ctr_gap ×
log1p(impressions_90d)`, with no fitted weights, and the tier-mean-CTR lookup fit on **training
rows only** so the baseline itself is evaluated out-of-sample.

**Model.** Random Forest on a **leakage-checked, safe feature set**: `avg_position`,
`content_age_days`, `days_since_last_update`, `word_count`, `char_count`, `search_volume`,
`competition`, `cpc`, plus content-type/intent/provider metadata and missingness flags.
`impressions_90d`, `clicks_90d`, `ctr`, and every other trailing-90-day engagement column are
**excluded** — see the leakage audit below. An earlier full-feature comparison (`w05_model.ipynb`)
found Logistic Regression competitive with (and on some metrics ahead of) Random Forest — Random
Forest is used here because it produced the strongest result on the final safe feature set, not
because it dominated on every cut of the data.

**Validation design.** An 80/20 **client-grouped** split — 6 of 32 clients held out entirely, zero
client overlap between train and test. `w06_validation_audit.ipynb` tested this directly against a
naive random row-level split on the same data: the random split let 31 of 32 clients appear in
*both* sides and inflated precision@20 to 0.95; the honest, client-grouped number is 0.70 for the
same baseline. The model was partly memorizing per-client base rates, not generalizing — a finding
in its own right, not just a caveat.

**Leakage checks (`w03_data_contract.ipynb`, `w06_validation_audit.ipynb`):**
1. *Classic trap* — deliberately feeding the label's own future-window input as a feature drove
   AUC to 1.000. Removed.
2. *Window-overlap check* — `impressions_90d` correlates at **r = 0.98** with the exact columns the
   label is computed from. An ablation (with vs. without the overlapping features) moved ROC-AUC by
   only −0.017 and *improved* precision@k — real enough to disclose and exclude, but not the
   dramatic near-1.0 collapse a classic leakage case shows.
3. *Harness sanity check* — smuggling `trend_pct` in as the only feature reproduced AUC = 1.000,
   confirming the audit would catch real leakage if it were there.

In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

# --- client-grouped 80/20 split, same convention as w05/w06/w07 ---
clients = df["client_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(clients)
n_test_clients = max(1, round(len(shuffled) * 0.2))
test_clients = set(shuffled[:n_test_clients])
test_mask = df["client_id"].isin(test_clients)
train_df = df[~test_mask].reset_index(drop=True)
test_df = df[test_mask].reset_index(drop=True)
print(f"{len(shuffled)} clients -> {n_test_clients} held out | zero overlap: "
      f"{len(set(train_df.client_id) & set(test_df.client_id)) == 0}")
print(f"train: {len(train_df):,} rows (decline {train_df['is_declining_label'].mean():.3f}) | "
      f"test: {len(test_df):,} rows (decline {test_df['is_declining_label'].mean():.3f})")

pos_bins = [0, 3, 10, 20, 50, np.inf]
pos_labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]

def add_pos_tier(frame):
    frame = frame.copy()
    frame["position_tier_fixed"] = "no_position_data"
    has_pos = frame["avg_position"] > 0
    frame.loc[has_pos, "position_tier_fixed"] = pd.cut(
        frame.loc[has_pos, "avg_position"], bins=pos_bins, labels=pos_labels
    ).astype(str)
    return frame

train_df, test_df = add_pos_tier(train_df), add_pos_tier(test_df)

# --- baseline: ML-07 rule, tier-mean CTR fit on train only ---
train_has_pos = train_df["avg_position"] > 0
tier_mean_ctr = train_df.loc[train_has_pos].groupby("position_tier_fixed")["ctr"].mean()

def baseline_score(frame):
    expected_ctr = frame["position_tier_fixed"].map(tier_mean_ctr)
    visible = (frame["impressions_90d"] >= 100).astype(int)
    reachable = ((frame["avg_position"] > 0) & (frame["avg_position"] <= 50)).astype(int)
    ctr_gap = (expected_ctr - frame["ctr"]).clip(lower=0).fillna(0)
    return visible * reachable * ctr_gap * np.log1p(frame["impressions_90d"])

test_df["baseline_score"] = baseline_score(test_df)
baseline_metrics = {
    "precision_at_20": precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 20),
    "precision_at_50": precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 50),
    "precision_at_100": precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 100),
    "roc_auc": roc_auc_score(test_df["is_declining_label"], test_df["baseline_score"]),
    "average_precision": average_precision_score(test_df["is_declining_label"], test_df["baseline_score"]),
}
print("\nBASELINE (train-only tier-mean CTR rule):", {k: round(v, 3) for k, v in baseline_metrics.items()})

32 clients -> 6 held out | zero overlap: True
train: 27,675 rows (decline 0.555) | test: 2,325 rows (decline 0.391)

BASELINE (train-only tier-mean CTR rule): {'precision_at_20': 0.7, 'precision_at_50': 0.64, 'precision_at_100': 0.65, 'roc_auc': 0.596, 'average_precision': 0.476}


In [4]:
FULL_NUMERIC = ["search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
SUSPECT = ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
SAFE_NUMERIC = [c for c in FULL_NUMERIC if c not in SUSPECT]
CATEGORICAL = ["competition_level", "content_type", "main_intent", "provider_used", "model_used", "position_tier_fixed"]
LOG_COLUMNS = ["search_volume", "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
               "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d"]
MISSING_FLAG_COLUMNS = ["word_count", "char_count", "search_volume", "competition", "cpc",
                         "main_intent", "competition_level", "provider_used", "model_used"]

def engineer(frame, numeric_cols):
    out = frame.copy()
    for col in MISSING_FLAG_COLUMNS:
        out[f"has_{col}"] = out[col].notna().astype(int)
    num = out[numeric_cols].apply(pd.to_numeric, errors="coerce")
    for col in LOG_COLUMNS:
        if col in num.columns:
            num[col] = np.log1p(num[col].clip(lower=0))
    num = num.replace([np.inf, -np.inf], np.nan).fillna(0)
    flags = out[[f"has_{c}" for c in MISSING_FLAG_COLUMNS]]
    cat = pd.get_dummies(out[CATEGORICAL].astype(str), dummy_na=False, dtype=float)
    return pd.concat([num, flags, cat], axis=1)

def fit_and_score(train_df, test_df, numeric_cols):
    train_feat, test_feat = engineer(train_df, numeric_cols), engineer(test_df, numeric_cols)
    train_feat, test_feat = train_feat.align(test_feat, join="left", axis=1, fill_value=0)
    y_train, y_test = train_df["is_declining_label"], test_df["is_declining_label"]
    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=8, min_samples_leaf=25,
                                 n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(train_feat, y_train)
    scores = rf.predict_proba(test_feat)[:, 1]
    return {
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "precision_at_50": precision_at_k(y_test, scores, 50),
        "precision_at_100": precision_at_k(y_test, scores, 100),
        "roc_auc": roc_auc_score(y_test, scores),
        "average_precision": average_precision_score(y_test, scores),
    }, rf

model_metrics, rf_safe = fit_and_score(train_df, test_df, SAFE_NUMERIC)
print("MODEL, safe/leakage-checked features:", {k: round(v, 3) for k, v in model_metrics.items()})

# --- leakage ablation: same split, WITH the excluded window-overlap features ---
model_full_metrics, _ = fit_and_score(train_df, test_df, FULL_NUMERIC)
print("[context] Random Forest WITH window-overlap features:", {k: round(v, 3) for k, v in model_full_metrics.items()})
print(f"ROC-AUC change from removing them: {model_metrics['roc_auc'] - model_full_metrics['roc_auc']:+.3f}")

# --- harness sanity check: smuggle trend_pct in alone ---
X_leak_train = train_df[["trend_pct"]].fillna(0).to_numpy()
X_leak_test = test_df[["trend_pct"]].fillna(0).to_numpy()
leak_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1)
leak_model.fit(X_leak_train, train_df["is_declining_label"])
leak_scores = leak_model.predict_proba(X_leak_test)[:, 1]
print(f"Sanity check -- trend_pct smuggled in as the ONLY feature: "
      f"ROC-AUC = {roc_auc_score(test_df['is_declining_label'], leak_scores):.4f} (expected ~1.0)")

MODEL, safe/leakage-checked features: {'precision_at_20': 0.75, 'precision_at_50': 0.72, 'precision_at_100': 0.66, 'roc_auc': 0.708, 'average_precision': 0.55}


[context] Random Forest WITH window-overlap features: {'precision_at_20': 0.7, 'precision_at_50': 0.66, 'precision_at_100': 0.52, 'roc_auc': 0.725, 'average_precision': 0.547}
ROC-AUC change from removing them: -0.017


Sanity check -- trend_pct smuggled in as the ONLY feature: ROC-AUC = 1.0000 (expected ~1.0)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Same held-out, client-grouped test split (2,325 pages across the 6 unseen clients, base rate
0.391) for both:

| Metric | Baseline rule | Model (safe features) |
|---|---:|---:|
| Precision@20 | 0.70 | 0.75 |
| Precision@50 | 0.64 | 0.72 |
| Precision@100 | 0.65 | 0.66 |
| ROC-AUC | 0.596 | 0.708 |
| Average precision | 0.476 | 0.550 |

**Three honest reads, not one flattering headline.** The baseline rule is a genuinely strong
top-of-list ranker — using nothing but three trailing-90-day numbers multiplied together, it's not
far behind the model at precision@20 and @100. The model's real advantage is breadth: ROC-AUC
(+0.112) and average precision (+0.074) show it separates decliners from non-decliners better
across the *whole* ranked list, picking up signal (position, content age, tier, metadata) the
three-input rule never looks at. Precision@100 barely moved (0.65 → 0.66) — the model's edge
concentrates near the top of the queue, exactly where reviewer capacity is scarcest, but that also
means "the model wins everywhere" would overstate the finding.

In [5]:
print(f"{'metric':<20}{'baseline':>12}{'model':>12}{'delta':>10}")
for k in ["precision_at_20", "precision_at_50", "precision_at_100", "roc_auc", "average_precision"]:
    b, m = baseline_metrics[k], model_metrics[k]
    print(f"{k:<20}{b:>12.3f}{m:>12.3f}{m-b:>+10.3f}")

metric                  baseline       model     delta
precision_at_20            0.700       0.750    +0.050
precision_at_50            0.640       0.720    +0.080
precision_at_100           0.650       0.660    +0.010
roc_auc                    0.596       0.708    +0.112
average_precision          0.476       0.550    +0.074


## 5. Limitations

*What this work cannot claim.*

- **The label is a backward-looking proxy, not a future-observed outcome.** It buckets a 60-day
  trailing comparison at export time — it answers "was this page already declining," not "will it
  decline next month." A genuine forward-window label was shown buildable from the full warehouse
  (ROC-AUC 0.775, `w03_data_contract.ipynb`) but needs gated access this paper's reproducible
  pipeline does not depend on.
- **No time-based holdout.** Validation tests generalization to unseen *clients*, never to *future*
  data. Seasonal effects, algorithm updates, or drift over time are untested.
- **No revenue data.** The recommendation queue's "value-at-stake" number is a directional,
  order-of-magnitude proxy from CPC and estimated recoverable clicks, not a dollar forecast.
- **A residual, disclosed leakage risk.** `impressions_90d`-family features structurally overlap
  the label's own defining window (r = 0.98) and are excluded from the model for exactly that
  reason, even though the ablation found their effect on the score modest, not dominant.
- **`no_position_data` (4.0% of pages) is a measurement gap, not a clean bill of health.** These
  pages show a 0.7% decline rate vs. 56.4% for pages with real position data — almost certainly a
  near-zero-traffic artifact, not evidence of health.
- **No causal claims, anywhere.** This is association/prediction on cross-sectional traffic data —
  observed / measured / directional / decision-support language throughout, never "caused" or
  "predicted Google's algorithm."
- **An internship-scale demonstration project**, not a validated production system. Nothing here
  should be presented to a client, or used for pricing, staffing, or contractual claims, without
  new, purpose-built validation.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Built in `w07_action_playbook.ipynb` from two signals kept deliberately separate: the descriptive
CTR-vs-position rule (today's underperformance vs. peers) and the predictive risk model (safe
features only). Every page gets a fixed reason code and one of five actions:

| Action | Meaning | Pages |
|---|---|---:|
| `refresh_and_review_ctr` | Highest priority — both signals agree | 5,092 |
| `refresh_content` | Model flags real decline risk; CTR isn't the (only) problem | 2,408 |
| `review_ctr` | Underperforming peers today; risk model doesn't (yet) agree | 11,966 |
| `monitor` | Neither signal fired | 9,329 |
| `no_action_insufficient_data` | No real position data — a tracking gap, route to data owner | 1,205 |

**Never automated:** auto-publishing or auto-editing content, auto-pausing a client's content
without human sign-off, client-facing or contractual claims from these numbers, or treating
`no_position_data` rows as low-risk just because the model scores them that way.

In [6]:
import json
with open("work/outputs/action_playbook_metrics.json") as f:
    committed_metrics = json.load(f)

print("Committed receipts (work/outputs/action_playbook_metrics.json):")
print(f"  held-out validation: {committed_metrics['held_out_client_grouped_validation']}")
print(f"  action mix: {committed_metrics['action_counts']}")
print(f"  archetype counts: {committed_metrics['archetype_counts']}")

# Confirm this notebook's freshly reproduced model metrics match the committed receipts
ref = committed_metrics["held_out_client_grouped_validation"]
match = all(abs(model_metrics[k] - ref[k]) < 0.005 for k in ["roc_auc", "precision_at_20", "precision_at_50", "precision_at_100"])
print(f"\nFreshly reproduced model metrics match the committed w07 receipts: {match}")

Committed receipts (work/outputs/action_playbook_metrics.json):
  held-out validation: {'base_rate': 0.391, 'roc_auc': 0.708, 'precision_at_20': 0.75, 'precision_at_50': 0.72, 'precision_at_100': 0.66}
  action mix: {'review_ctr': 11966, 'monitor': 9329, 'refresh_and_review_ctr': 5092, 'refresh_content': 2408, 'no_action_insufficient_data': 1205}
  archetype counts: {'steady_performer': 8751, 'hidden_gem': 8107, 'at_risk_visible': 5092, 'ctr_watch': 3859, 'quiet_decliner': 2408, 'data_quality_gap': 1205, 'champion': 578}

Freshly reproduced model metrics match the committed w07 receipts: True


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper (`docs/index.html`) embeds three figures, inlined as SVG so the page stays a
single self-contained file with no broken relative-path images:

1. **Results comparison** — precision@20/50/100, baseline vs. model, generated fresh from this
   notebook's own reproduced numbers (matches Section 4's table above).
2. **`work/figures/decay_refresh_insight.svg`** — the MIXED staleness-vs-decline-rate signal test,
   committed in `w07_action_playbook.ipynb`, reused unchanged as the "negative result kept in the
   record" callout in the paper's Methodology section.
3. **`work/figures/action_mix.svg`** — the five-action queue breakdown, committed in
   `w07_action_playbook.ipynb`, reused unchanged in the paper's Recommendations section.

The results-comparison chart is regenerated here (not just copy-pasted) so the artifact and the
numbers above are guaranteed to agree.

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/figures", exist_ok=True)

ks = ["precision@20", "precision@50", "precision@100"]
baseline_vals = [baseline_metrics["precision_at_20"], baseline_metrics["precision_at_50"], baseline_metrics["precision_at_100"]]
model_vals = [model_metrics["precision_at_20"], model_metrics["precision_at_50"], model_metrics["precision_at_100"]]
test_base_rate = float(test_df["is_declining_label"].mean())

fig, ax = plt.subplots(figsize=(7.2, 4.3))
x = np.arange(len(ks))
w = 0.32
b1 = ax.bar(x - w/2, baseline_vals, width=w, label="baseline rule (ML-07)", color="#9AA3B2")
b2 = ax.bar(x + w/2, model_vals, width=w, label="model, safe features (ML-09)", color="#2F5D50")
ax.axhline(test_base_rate, color="#B23A48", linestyle="--", linewidth=1.2, label=f"base rate ({test_base_rate:.2f})")
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.2f}", (bar.get_x() + bar.get_width()/2, h), textcoords="offset points",
                    xytext=(0, 4), ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(ks)
ax.set_ylabel("precision (share of top-K truly declining)")
ax.set_ylim(0, 0.95)
ax.set_title("Model vs. baseline on the same held-out, client-grouped test split")
ax.legend(fontsize=9, loc="upper right", frameon=False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig("work/figures/results_comparison.svg")
plt.show()
print("Saved work/figures/results_comparison.svg -- embedded directly (inlined) in docs/index.html")

Saved work/figures/results_comparison.svg -- embedded directly (inlined) in docs/index.html


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `client_id`/`content_id`
      values, same as every other week
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and
      **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut +
      a 3-sentence employer-facing summary. (Below.)

---

## ML-12 — Closing: demo, social cut, employer summary

### 5-minute demo outline
1. **(30s) The backlog problem.** 54.2% of pages trending down; at 20 reviews/week that's a
   658-week backlog. Reviewer attention is the scarce resource — ranking, not raw accuracy, is the
   right frame.
2. **(60s) The transparent baseline.** Walk through the CTR-vs-position rule and why staleness was
   tested and rejected (MIXED signal) before CTR (CONFIRMED) was chosen — show
   `decay_refresh_insight.svg`.
3. **(90s) The honest validation story.** Show the client-grouped vs. random-split before/after
   (0.95 vs. 0.70 precision@20) — this is the single most persuasive five minutes of evidence that
   the leakage/validation discipline was real, not decorative.
4. **(60s) The results table.** Baseline vs. model, same split, same metric — model wins at the top
   of the list and on ROC-AUC, but not everywhere. Show `results_comparison.svg`.
5. **(60s) The recommendations, live.** Walk the five-action queue and the "never automate" list —
   land on the point that this ranks, it doesn't decide.

### Social-post cut
> Built and shipped a client-grouped, leakage-checked model that beats a transparent CTR-vs-position
> rule at ranking which content pages a team should review first — ROC-AUC 0.596 → 0.708, on FlyRank's
> real search-performance data (30K pages, 32 clients). The honest finding: a naive random split would
> have overstated precision@20 by 25 points (0.95 vs. 0.70) — the validation discipline mattered more
> than the model choice. Full paper + reproducible repo: [link].

### Employer-facing summary (3 sentences)
I built a content-prioritization model on FlyRank's real search-performance data (30,000 pages, 32
clients), comparing a transparent baseline rule against a Random Forest under a client-grouped
holdout designed to catch exactly the kind of leakage and overfitting that make results look better
than they are. The model shows a real, moderate lift over the baseline (precision@50: 0.64 → 0.72;
ROC-AUC: 0.596 → 0.708) — and the project's most valuable finding was arguably methodological: a
naive random split would have overstated the same model's precision@20 by 25 points. The full
pipeline, leakage audits, and a five-action recommendation playbook are documented and reproducible
end to end in the linked repo.